# Stage 6: Choose one expert per dataset, then combine them

**Stage:** 6 (selection + ensemble) · **Runs on:** Kaggle and locally (CPU) · uses **validation data only**

1. For each dataset, the best of its 5 checkpoints (by macro F1 on that dataset's validation part)
   becomes that dataset's **expert**.
2. The four experts each give P(misleading) for every validation text of every dataset. Three
   combiners are compared: **soft vote** (average), **weighted average** (weights = each expert's
   validation F1) and a **blender** (Logistic Regression on the four probabilities, each dataset
   weighted equally, scored with 5-fold cross-validation).
3. The best combiner (by macro F1 averaged over the four datasets) becomes the final classifier,
   which the website uses.
4. Regression: the average of Ridge and LightGBM is added, and the best regression model is chosen
   by validation MAE.

| | |
|---|---|
| Input | Stage 2 split files, all Stage 5 prediction files |
| Output | `final_model.json`, `blender.joblib`, `pred__final.csv`, `reg_final.json`, `pred__average_reg__liar.csv` |

**On Kaggle:** attach the `misinfo_utils` utility script and add these notebooks as inputs (**Add Input → Your Work → Notebooks**): `02_split` and every Stage 5 notebook (05a–05g).

In [7]:
# Setup: import the shared code. Works on Kaggle and locally.
import sys
from pathlib import Path

# Locally, misinfo_utils.py is in <project>/src/. On Kaggle it is an attached Utility Script,
# so this loop finds nothing and the import below still works.
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src" / "misinfo_utils.py").exists():
        sys.path.insert(0, str(folder / "src"))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import misinfo_utils as mu

sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 100)
paths = mu.setup("06")

misinfo_utils | stage 06 | running on local
  read from : D:\misinfo-project\data\raw, D:\misinfo-project\outputs
  write to  : D:\misinfo-project\outputs\06_ensemble


In [8]:
import json, joblib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

labels = {(ds, s): mu.load_split(ds, s)[["row_id", "label"]] for ds in mu.DATASETS for s in ("val", "test")}

# Models with predictions for all four datasets (DistilBERT is skipped if it was not run)
available = [m for m in mu.CLF_MODELS
             if all(mu.find_file(mu.pred_file(m, ds), required=False) for ds in mu.DATASETS)]
preds = {(m, ds): mu.load_predictions(m, ds) for m in available for ds in mu.DATASETS}
print("Models found:", available)


def probs(model, trained_on, tested_on, split):
    # P(misleading) from one checkpoint, in the same row order as the split file
    df = preds[(model, trained_on)]
    df = df[(df["tested_on"] == tested_on) & (df["split"] == split)].set_index("row_id")["prob_misleading"]
    return df.loc[labels[(tested_on, split)]["row_id"]].values

Models found: ['logreg', 'complementnb', 'linearsvm', 'lightgbm', 'distilbert']


## 1. Choose one expert per dataset

In [9]:
selection = pd.DataFrame({
    m: {ds: f1_score(labels[(ds, "val")]["label"].astype(int), (probs(m, ds, ds, "val") >= 0.5).astype(int),
                     average="macro") for ds in mu.DATASETS}
    for m in available}).loc[mu.DATASETS].round(4)
experts = {ds: selection.loc[ds].idxmax() for ds in mu.DATASETS}
selection.columns = [mu.MODEL_NAMES[m] for m in selection.columns]
selection["expert"] = [mu.MODEL_NAMES[experts[ds]] for ds in mu.DATASETS]
selection   # validation macro F1 of every checkpoint on its own dataset; table for the report

,Logistic Regression,Complement NB,Linear SVM,LightGBM,DistilBERT,expert
liar,0.6141,0.6275,0.6185,0.6094,0.6605,DistilBERT
welfake,0.9469,0.8990,0.9499,0.9440,0.9764,DistilBERT
covid,0.9284,0.9181,0.9270,0.9070,0.9446,DistilBERT
fakenewsnet,0.7852,0.7873,0.7839,0.7678,0.7968,DistilBERT


## 2. Combine the four experts

In [10]:
def expert_matrix(split):
    # Rows = every text of every dataset's split; columns = the four experts (in mu.DATASETS order)
    P, y, groups, ids = [], [], [], []
    for tested in mu.DATASETS:
        P.append(np.column_stack([probs(experts[e], e, tested, split) for e in mu.DATASETS]))
        y.append(labels[(tested, split)]["label"].astype(int).values)
        groups += [tested] * len(y[-1])
        ids += labels[(tested, split)]["row_id"].tolist()
    return np.vstack(P), np.concatenate(y), np.array(groups), ids


V, y_val, g_val, ids_val = expert_matrix("val")
T, y_test, g_test, ids_test = expert_matrix("test")

# Each dataset counts equally in the blender, whatever its size
sample_weight = pd.Series(g_val).map(1 / pd.Series(g_val).value_counts()).values

weights = {ds: float(selection.loc[ds, mu.MODEL_NAMES[experts[ds]]]) for ds in mu.DATASETS}
w = np.array([weights[ds] for ds in mu.DATASETS])

# Blender scored fairly: each validation row is predicted by a blender NOT trained on it
oof = np.zeros(len(y_val))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=mu.SEED)
for tr, te in cv.split(V, pd.Series(g_val) + "_" + pd.Series(y_val).astype(str)):
    fold = LogisticRegression(max_iter=1000).fit(V[tr], y_val[tr], sample_weight=sample_weight[tr])
    oof[te] = fold.predict_proba(V[te])[:, 1]

val_probs = {"soft": V.mean(axis=1), "weighted": V @ (w / w.sum()), "blender": oof}
comparison = pd.DataFrame({
    name: {"mean macro F1 over datasets": mu.mean_dataset_f1(y_val, p, g_val),
           "macro F1, all rows pooled": f1_score(y_val, (p >= 0.5).astype(int), average="macro"),
           **{f"macro F1 {ds}": f1_score(y_val[g_val == ds], (p[g_val == ds] >= 0.5).astype(int), average="macro")
              for ds in mu.DATASETS}}
    for name, p in val_probs.items()}).T.round(4)
best = comparison["mean macro F1 over datasets"].idxmax()
display(comparison)
print("Chosen combiner:", best)

,mean macro F1 over datasets,"macro F1, all rows pooled",macro F1 liar,macro F1 welfake,macro F1 covid,macro F1 fakenewsnet
soft,0.6580,0.6970,0.5915,0.7580,0.8635,0.4191
weighted,0.6628,0.7096,0.5732,0.7989,0.8614,0.4179
blender,0.6845,0.7568,0.4622,0.9655,0.7760,0.5343


Chosen combiner: blender


In [11]:
blender = LogisticRegression(max_iter=1000).fit(V, y_val, sample_weight=sample_weight)
joblib.dump(blender, mu.output_path("blender.joblib"))
print("Blender weights (how much it trusts each expert):")
print(pd.Series(blender.coef_[0], index=[f"{ds} expert ({mu.MODEL_NAMES[experts[ds]]})" for ds in mu.DATASETS]).round(3).to_string())

config = {"experts": experts, "order": mu.DATASETS, "combiner": best, "weights": weights,
          "blender_file": "blender.joblib", "chosen_by": "validation macro F1 averaged over the four datasets"}
with open(mu.output_path(mu.FINAL_CONFIG), "w") as f:
    json.dump(config, f, indent=2)

test_p = {"soft": T.mean(axis=1), "weighted": T @ (w / w.sum()), "blender": blender.predict_proba(T)[:, 1]}[best]
final = pd.concat([
    pd.DataFrame({"row_id": ids_val, "tested_on": g_val, "split": "val", "prob_misleading": val_probs[best]}),
    pd.DataFrame({"row_id": ids_test, "tested_on": g_test, "split": "test", "prob_misleading": test_p})])
mu.save_csv(final, "pred__final.csv")
print(json.dumps(config, indent=2))

Blender weights (how much it trusts each expert):
liar expert (DistilBERT)           0.088
welfake expert (DistilBERT)        0.186
covid expert (DistilBERT)          0.217
fakenewsnet expert (DistilBERT)    0.158
Saved  15,239 rows -> pred__final.csv
{
  "experts": {
    "liar": "distilbert",
    "welfake": "distilbert",
    "covid": "distilbert",
    "fakenewsnet": "distilbert"
  },
  "order": [
    "liar",
    "welfake",
    "covid",
    "fakenewsnet"
  ],
  "combiner": "blender",
  "weights": {
    "liar": 0.6605,
    "welfake": 0.9764,
    "covid": 0.9446,
    "fakenewsnet": 0.7968
  },
  "blender_file": "blender.joblib",
  "chosen_by": "validation macro F1 averaged over the four datasets"
}


## 3. Regression: add the average and choose the best model

In [12]:
reg = {m: mu.load_predictions(m, "liar") for m in mu.REG_MODELS}
y_reg = {s: mu.load_split("liar_reg", s).set_index("row_id")["score"].astype(float) for s in ("val", "test")}

def reg_pred(m, split):
    return reg[m].query("split == @split").set_index("row_id")["pred_score"].loc[y_reg[split].index]

average = pd.concat([pd.DataFrame({"row_id": y_reg[s].index, "split": s,
                                   "pred_score": (reg_pred("ridge_reg", s).values + reg_pred("lightgbm_reg", s).values) / 2})
                     for s in ("val", "test")])
mu.save_csv(average, mu.pred_file("average_reg", "liar"))
reg["average_reg"] = average

reg_val = pd.DataFrame({mu.MODEL_NAMES[m]: mu.reg_scores(y_reg["val"], reg_pred(m, "val"))
                        for m in mu.REG_MODELS + ["average_reg"]}).T.round(3)
best_reg = min(["ridge_reg", "lightgbm_reg", "average_reg"],
               key=lambda m: mu.reg_scores(y_reg["val"], reg_pred(m, "val"))["MAE"])
with open(mu.output_path("reg_final.json"), "w") as f:
    json.dump({"best": best_reg, "chosen_by": "validation MAE"}, f, indent=2)
display(reg_val)
print("Best regression model:", mu.MODEL_NAMES[best_reg])

Saved   3,838 rows -> pred__average_reg__liar.csv


,MAE,RMSE,R2,within_1_level_%
Dummy (mean),1.348,1.561,-0.000,37.018
Ridge,1.268,1.495,0.083,41.971
LightGBM regressor,1.290,1.521,0.051,40.667
Average (Ridge + LightGBM),1.272,1.495,0.083,41.554


Best regression model: Ridge
